# 02 — Bronze (Auto Loader → VARIANT Delta)

Reads every JSON file landed in the Volume by notebook 01 into a small set of
**VARIANT-typed** bronze tables. One run is `availableNow=True`, so re-runs
just pick up files added since the last execution (idempotent).

| Bronze table | Source folder in Volume | Grain |
|--------------|-------------------------|-------|
| `bronze_competitions` | `reference/competitions*.json` | One per competition |
| `bronze_teams` | `reference/teams.json` | One per call |
| `bronze_team_records` | `reference/team_records.json` | One per call |
| `bronze_venues` | `reference/venues.json` | One per call |
| `bronze_players` | `reference/players_*.json` | One per season-level pull |
| `bronze_metric_topics` | `reference/metric_topics_*.json` | One per scope |
| `bronze_game` | `games/*/game.json` | One per game |
| `bronze_game_rosters` | `games/*/roster.json` | One per game |
| `bronze_compiled_events` | `games/*/compiled_events.json` | One per game |
| `bronze_full_events` | `games/*/full_events.json` | One per game |
| `bronze_shift_events` | `games/*/shift_events.json` | One per game |
| `bronze_player_toi` | `games/*/player_toi.json` | One per game |
| `bronze_game_metrics` | `games/*/metrics_player_*.json` | One per game per topic |
| `bronze_season_metrics` | `season_metrics/*/*.json` | One per scope-topic |
| `bronze_xrefs` | `xrefs/**/*.json` | One per (system, entity) |
| `bronze_player_history` | `player_history/*.json` | One per sampled player |

Every bronze row carries:
- `data VARIANT` — the full raw payload, navigable with `data:field::type`
- `_ingestion_timestamp` — when the row landed in bronze
- `_source_file` — the Volume path it came from

In [1]:
import os, json
from dotenv import load_dotenv
load_dotenv()

# Dual-mode Spark session — works locally via Databricks Connect AND inside a
# Databricks workspace notebook. In the workspace, `spark` already exists.
try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

In [2]:
UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "sportlogiq_nhl")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"
VOLUME_NAME   = "raw_data"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"

print(f"Catalog: {UC_CATALOG}")
print(f"Bronze:  {UC_CATALOG}.{BRONZE_SCHEMA}")
print(f"Silver:  {UC_CATALOG}.{SILVER_SCHEMA}")
print(f"Gold:    {UC_CATALOG}.{GOLD_SCHEMA}")
print(f"Volume:  {VOLUME_PATH}")

Catalog: alexander_booth
Bronze:  alexander_booth.sportlogiq_nhl_bronze
Silver:  alexander_booth.sportlogiq_nhl_silver
Gold:    alexander_booth.sportlogiq_nhl_gold
Volume:  /Volumes/alexander_booth/sportlogiq_nhl_bronze/raw_data


In [3]:
ENTITIES = [
    # (bronze_table_suffix, glob_path under raw_data)
    # `competition*.json` matches both the list (`competitions.json`) and the
    # detail (`competition_<id>.json`) — silver filters on data:id IS NOT NULL.
    ("competitions",     "reference/competition*.json"),
    ("teams",            "reference/teams.json"),
    ("team_records",     "reference/team_records.json"),
    ("venues",           "reference/venues.json"),
    ("players",          "reference/players_*.json"),
    ("metric_topics",    "reference/metric_topics_*.json"),
    ("game",             "games/*/game.json"),
    ("game_rosters",     "games/*/roster.json"),
    ("compiled_events",  "games/*/compiled_events.json"),
    ("full_events",      "games/*/full_events.json"),
    ("shift_events",     "games/*/shift_events.json"),
    ("player_toi",       "games/*/player_toi.json"),
    ("game_metrics",     "games/*/metrics_player_*.json"),
    ("season_metrics",   "season_metrics/*/*.json"),
    ("xrefs",            "xrefs/*/*.json"),
    ("player_history",   "player_history/*.json"),
]
print(f"Bronze entities to load: {len(ENTITIES)}")

Bronze entities to load: 16


In [4]:
for entity, glob in ENTITIES:
    source_path     = f"{VOLUME_PATH}/{glob.rsplit('/', 1)[0]}/"
    glob_filter     = glob.rsplit('/', 1)[1]  # filename pattern, e.g. "metrics_player_*.json"
    checkpoint_path = f"{VOLUME_PATH}/_checkpoints/{entity}"
    target_table    = f"{UC_CATALOG}.{BRONZE_SCHEMA}.bronze_{entity}"

    print(f"\n[{entity}] -> {target_table}")
    print(f"  source: {source_path}{glob_filter}")

    try:
        (
            spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "json")
            .option("cloudFiles.inferColumnTypes", "true")
            .option("cloudFiles.schemaLocation", checkpoint_path + "/_schema")
            .option("cloudFiles.schemaEvolutionMode", "rescue")
            .option("pathGlobFilter", glob_filter)
            .option("multiLine", "true")
            .load(source_path)
            .selectExpr(
                "PARSE_JSON(TO_JSON(STRUCT(*))) AS data",
                "current_timestamp() AS _ingestion_timestamp",
                "_metadata.file_path AS _source_file",
            )
            .writeStream
            .format("delta")
            .outputMode("append")
            .option("checkpointLocation", checkpoint_path)
            .trigger(availableNow=True)
            .toTable(target_table)
        ).awaitTermination()
        n = spark.table(target_table).count()
        print(f"  done ({n:,} rows total)")
    except Exception as e:
        # Empty source folders are common during first-time partial ingests; surface but don't halt.
        print(f"  skipped: {str(e)[:160]}")


[competitions] -> alexander_booth.sportlogiq_nhl_bronze.bronze_competitions
  source: /Volumes/alexander_booth/sportlogiq_nhl_bronze/raw_data/reference/competition*.json


  done (1 rows total)

[teams] -> alexander_booth.sportlogiq_nhl_bronze.bronze_teams
  source: /Volumes/alexander_booth/sportlogiq_nhl_bronze/raw_data/reference/teams.json


  done (1 rows total)

[team_records] -> alexander_booth.sportlogiq_nhl_bronze.bronze_team_records
  source: /Volumes/alexander_booth/sportlogiq_nhl_bronze/raw_data/reference/team_records.json


  done (1 rows total)

[venues] -> alexander_booth.sportlogiq_nhl_bronze.bronze_venues
  source: /Volumes/alexander_booth/sportlogiq_nhl_bronze/raw_data/reference/venues.json


  done (1 rows total)

[players] -> alexander_booth.sportlogiq_nhl_bronze.bronze_players
  source: /Volumes/alexander_booth/sportlogiq_nhl_bronze/raw_data/reference/players_*.json


...

  done (120 rows total)

[game_metrics] -> alexander_booth.sportlogiq_nhl_bronze.bronze_game_metrics
  source: /Volumes/alexander_booth/sportlogiq_nhl_bronze/raw_data/games/*/metrics_player_*.json


  done (10 rows total)

[season_metrics] -> alexander_booth.sportlogiq_nhl_bronze.bronze_season_metrics
  source: /Volumes/alexander_booth/sportlogiq_nhl_bronze/raw_data/season_metrics/*/*.json


  done (8 rows total)

[xrefs] -> alexander_booth.sportlogiq_nhl_bronze.bronze_xrefs
  source: /Volumes/alexander_booth/sportlogiq_nhl_bronze/raw_data/xrefs/*/*.json


  skipped: (com.databricks.sql.cloudfiles.errors.CloudFilesException) [CF_EMPTY_DIR_FOR_SCHEMA_INFERENCE] Cannot infer schema when the input path `/Volumes/alexander_booth

[player_history] -> alexander_booth.sportlogiq_nhl_bronze.bronze_player_history
  source: /Volumes/alexander_booth/sportlogiq_nhl_bronze/raw_data/player_history/*.json


  skipped: (com.databricks.sql.cloudfiles.errors.CloudFilesException) [CF_EMPTY_DIR_FOR_SCHEMA_INFERENCE] Cannot infer schema when the input path `/Volumes/alexander_booth


## Verify

In [5]:
print("=" * 60)
print("BRONZE LAYER SUMMARY")
print("=" * 60)
rows = spark.sql(f"""
    SELECT table_name, COUNT(*) FROM (
      SELECT '{UC_CATALOG}.{BRONZE_SCHEMA}.' || table_name AS table_name
      FROM {UC_CATALOG}.information_schema.tables
      WHERE table_schema = '{BRONZE_SCHEMA}' AND table_name LIKE 'bronze_%'
    ) GROUP BY 1 ORDER BY 1
""").collect()

for entity, _ in ENTITIES:
    table = f"{UC_CATALOG}.{BRONZE_SCHEMA}.bronze_{entity}"
    try:
        n = spark.table(table).count()
        print(f"  {table:<70s}  {n:>10,} rows")
    except Exception as e:
        print(f"  {table:<70s}  (missing — {str(e)[:60]})")

print("\nSample bronze_game (raw VARIANT navigation):")
spark.sql(f"""
    SELECT data:id::string AS game_id,
           data:season::string AS season,
           data:date::string AS game_date,
           data:home_team.shorthand::string AS home,
           data:away_team.shorthand::string AS away,
           _ingestion_timestamp
    FROM {UC_CATALOG}.{BRONZE_SCHEMA}.bronze_game
    LIMIT 5
""").show(truncate=False)

BRONZE LAYER SUMMARY


  alexander_booth.sportlogiq_nhl_bronze.bronze_competitions                        1 rows


  alexander_booth.sportlogiq_nhl_bronze.bronze_teams                               1 rows


  alexander_booth.sportlogiq_nhl_bronze.bronze_team_records                        1 rows


  alexander_booth.sportlogiq_nhl_bronze.bronze_venues                              1 rows


...

  alexander_booth.sportlogiq_nhl_bronze.bronze_game_metrics                       10 rows


  alexander_booth.sportlogiq_nhl_bronze.bronze_season_metrics                      8 rows


  alexander_booth.sportlogiq_nhl_bronze.bronze_xrefs                      (missing — [TABLE_OR_VIEW_NOT_FOUND] The table or view `alexander_booth)


  alexander_booth.sportlogiq_nhl_bronze.bronze_player_history             (missing — [TABLE_OR_VIEW_NOT_FOUND] The table or view `alexander_booth)

Sample bronze_game (raw VARIANT navigation):


+-------+--------+----------+----+----+-----------------------+
|game_id|season  |game_date |home|away|_ingestion_timestamp   |
+-------+--------+----------+----+----+-----------------------+
|1003   |20232024|2023-10-20|NYR |TBL |2026-05-01 15:54:55.614|
|1004   |20232024|2023-10-22|VGK |COL |2026-05-01 15:54:55.614|
|1005   |20232024|2023-10-25|COL |CHI |2026-05-01 15:54:55.614|
|1002   |20232024|2023-10-17|BOS |TOR |2026-05-01 15:54:55.614|
|1001   |20232024|2023-10-15|BUF |BOS |2026-05-01 15:54:55.614|
+-------+--------+----------+----+----+-----------------------+

